# SST thermal simulation

Run the coupled electromagnetic and thermal COMSOL study with Stationary Solver 2 configured for iterative FGMRES.

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from pathlib import Path
import mph

Start the COMSOL client and load the model

In [14]:
client = mph.start()

In [15]:
model_file = Path('E_quartercore_thermal_initial.mph').resolve()
if not model_file.is_file():
    raise FileNotFoundError(model_file)
model = client.load(model_file)

Select and label the coupled electromagnetic and thermal study

In [16]:
study_name = 'Study (EM + Thermal)'
studies = model.studies()
if study_name not in studies:
    if len(studies) != 1:
        raise ValueError(f'Expected one study to rename, found: {studies}')
    (model / 'studies' / studies[0]).rename(study_name)
study_node = model / 'studies' / study_name
if not study_node.exists():
    raise RuntimeError(f'Could not select study: {study_name}')
print('Selected study:', study_name)

Selected study: Study (EM + Thermal)


Configure Stationary Solver 2 to use Iterative 1 with FGMRES

In [17]:
if 'model' not in globals():
    raise RuntimeError('Run the model-loading cells first.')

# Resolve the study in this cell so it also works after individual cell reruns.
study_name = 'Study (EM + Thermal)'
studies = model.studies()
if study_name not in studies:
    if len(studies) != 1:
        raise ValueError(f'Expected one study to rename, found: {studies}')
    (model / 'studies' / studies[0]).rename(study_name)
study_node = model / 'studies' / study_name
if not study_node.exists():
    raise RuntimeError(f'Could not resolve study: {study_name}')

# The initial MPH file does not store a solver configuration. Generate the
# study-controlled default sequence before changing Stationary Solver 2.
solution = model / 'solutions' / 'Solution 1'
stationary_solver = solution / 'Stationary Solver 2'
if not stationary_solver.exists():
    print('Generating the default COMSOL solver configuration...')
    java_study = study_node.java_if_exists()
    create_auto_sequences = getattr(java_study, 'createAutoSequences', None)
    if create_auto_sequences is None:
        raise RuntimeError('This COMSOL study does not expose createAutoSequences().')
    create_auto_sequences('sol')

# Resolve the nodes after generation because they did not exist beforehand.
solution = model / 'solutions' / 'Solution 1'
stationary_solver = solution / 'Stationary Solver 2'
fully_coupled = stationary_solver / 'Fully Coupled 1'
iterative_solver = stationary_solver / 'Iterative 1'
required_nodes = [solution, stationary_solver, fully_coupled, iterative_solver]
missing = [node.name() for node in required_nodes if not node.exists()]
if missing:
    raise LookupError(f'COMSOL did not generate the expected solver nodes: {missing}')

# Match the requested Stationary Solver 2 configuration before solving.
iterative_solver.property('linsolver', 'fgmres')
iterative_solver.property('itrestart', 50)
iterative_solver.property('nlinnormuse', 'on')
iterative_solver.property('nlinnormlevel', 0.1)
iterative_solver.property('maxlinit', 10000)
fully_coupled.property('linsolver', iterative_solver.tag())
print('Study:', study_name)
print('Stationary Solver 2 linear solver:', iterative_solver.property('linsolver'))
print('Fully Coupled 1 uses solver tag:', fully_coupled.property('linsolver'))

Generating the default COMSOL solver configuration...
Study: Study (EM + Thermal)
Stationary Solver 2 linear solver: fgmres
Fully Coupled 1 uses solver tag: i1


Run the coupled electromagnetic and thermal simulation

In [18]:
if 'model' not in globals() or 'study_name' not in globals():
    raise RuntimeError('Run the notebook cells from the top first.')
model.solve(study_name)

Save the model with solution data

In [19]:
output_file = Path('E_quartercore_thermal_solved.mph').resolve()
model.save(output_file)
print('Saved:', output_file)

Saved: C:\Users\abujazar\github_repos\MPhSweepKit\examples\project_sst\thermal\quarter_block_winding\E_quartercore_thermal_solved.mph


Release the COMSOL client

In [20]:
client.remove(model)
client.clear()
client.disconnect()